In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pylab as plt

import seaborn as sns

from skspatial.objects import Line, Plane
from skspatial.plotting import plot_3d


from skspatial.objects import Line, Cylinder, Point, Points
from skspatial.plotting import plot_3d

import phasespace

import tensorflow

import bisect
import numpy as np
import matplotlib.pylab as plt
import pandas as pd

import seaborn as sns

import numpy as np
from sklearn.mixture import GaussianMixture
from scipy.stats import multivariate_normal

import numpy as np
from scipy.interpolate import griddata
from scipy.integrate import quad, trapezoid
from scipy.interpolate import CubicSpline

import matplotlib.pylab as plt
from scipy import stats
from matplotlib import cm
from matplotlib.ticker import LinearLocator

from scipy.interpolate import LinearNDInterpolator

import eloss_tools


import dm_generation_tools as dgt
import detector_simulation_tools as dst

import glob

import time

####################################
import warnings
# Suppress all warnings
warnings.filterwarnings("ignore")


import pickle

In [ ]:
def kinematic_diagnostic(d=-7.5, r=500, tag='mDM_200-10000_mA_0.22_dm_model_floating', masses=None):

    tag = f'd_{d}_r_{r}_{tag}'
        
    infile = f'generated_data_{tag}.parquet'
    df_decays = pd.read_parquet(infile)

    if masses is None:
        masses = df_decays['M_DM'].unique()

    df_decays['rho0'] = np.sqrt(df_decays['x0']**2 + df_decays['y0']**2) 

    for mass in masses:
        #mass = 1000
        
        filter = (df_decays['efinal_mu1']>1)
        filter = filter & (df_decays['M_DM']==mass)

        # For comparison
        filter = filter & (np.abs(df_decays['e_mu1']-mass/2)/mass < 0.01)

        
        plt.figure(figsize=(12,8))
        
        plt.subplot(2,2,1)
        plt.hist(df_decays[filter]['z0'],bins=100, range=(-4000,0))
        plt.xlabel('depth (m)', fontsize=18)

        plt.subplot(2,2,2)
        plt.hist(df_decays[filter]['rho0'],bins=100, range=(-10,500))
        plt.xlabel('radial distance (m)', fontsize=18)
        
        plt.subplot(2,2,3)
        df_decays[filter].plot.scatter(y='y0', x='x0', s=0.1, alpha=0.1, ax=plt.gca())        
        
        plt.subplot(2,2,4)
        #df_decays[filter].plot.scatter(y='efinal_mu1', x='y0', s=0.1, ax=plt.gca())
        df_decays[filter].plot.scatter(y='pt1_detector_acceptance_eloss', x='rho0', s=0.1, ax=plt.gca())
        
        DMstr = 'DM'
        lbracket = '{'
        rbracket = '}'
        plt.gcf().suptitle(f'$M_{lbracket}DM{rbracket}$ {int(mass)} GeV/c$^2$')
        
        plt.tight_layout()
        
        #outfile = 'depth_and_pt_d_{r}_r_{r}_{tag}.png'
        #plt.savefig(outfile)

        ########################################################################
        plt.figure(figsize=(12,4))

        plt.subplot(1,3,1)
        df_decays[filter]['e_mu1'].hist(bins=50, range=(-100,2000),  histtype="step",linewidth=2.5, label='Orig. energy')
        df_decays[filter]['efinal_mu1'].hist(bins=50, range=(-100,2000), histtype="step",linewidth=2.5,  label='Energy at detector')
        plt.xlabel(r'$E_{\mu}$ (GeV)', fontsize=18)
        plt.legend()

        plt.subplot(1,3,2)
        df_decays[filter]['e_mu1'].hist(bins=50, range=(-100,2000), density=True,  histtype="step",linewidth=2.5, label='Orig. energy')
        df_decays[filter]['efinal_mu1'].hist(bins=50, range=(-100,2000), density=True, histtype="step",linewidth=2.5,  label='Energy at detector')
        plt.xlabel(r'$E_{\mu}$ (GeV)', fontsize=18)
        plt.legend()
        plt.yscale('log')
        plt.ylim(5e-4)

        plt.subplot(1,3,3)
        df_decays[filter].plot.scatter(y='efinal_mu1', x='rho0', s=0.1, ax=plt.gca())
        plt.xlabel(r'Radial distance (m)', fontsize=18)
        plt.ylabel(r'$E_{\mu}$ (GeV)', fontsize=18)

        plt.tight_layout()

        
        ########################################################################
        plt.figure(figsize=(12,4))

        plt.subplot(1,3,1)
        df_decays[filter]['pt1_detector_acceptance'].hist(bins=50, range=(-100,2000),  histtype="step",linewidth=2.5, label='Ignoring eloss')
        df_decays[filter]['pt1_detector_acceptance_eloss'].hist(bins=50, range=(-100,2000), histtype="step",linewidth=2.5,  label='With eloss')
        plt.xlabel(r'$p_{T}$ (GeV)', fontsize=18)
        plt.legend()

        plt.subplot(1,3,2)
        df_decays[filter]['pt1_detector_acceptance'].hist(bins=50, range=(-100,2000),  histtype="step", density=True, linewidth=2.5, label='Ignoring eloss')
        df_decays[filter]['pt1_detector_acceptance_eloss'].hist(bins=50, range=(-100,2000), histtype="step",density=True, linewidth=2.5,  label='With eloss')
        plt.xlabel(r'$p_{T}$ (GeV)', fontsize=18)
        plt.legend()
        plt.yscale('log')
        
        plt.subplot(1,3,3)
        df_decays[filter].plot.scatter(y='pt1_detector_acceptance', x='rho0', s=0.1, ax=plt.gca())
        plt.xlabel(r'Radial distance (m)', fontsize=18)
        plt.ylabel(r'$p_{T}$ (GeV) (no eloss)', fontsize=18)

        plt.tight_layout()

        outfile = f'depth_and_pt_d_{r}_r_{r}_{tag}.png'
        plt.savefig(outfile)


    
    return df_decays
##################################

#df_decays = kinematic_diagnostic(masses=[2000])
#df_decays = kinematic_diagnostic(masses=[2000], d=0, r=10, tag='mDM_2000-3000_mA_0.22_dm_model_floating')
#df_decays = kinematic_diagnostic(masses=[2000], d=100, r=10, tag='mDM_2000-3000_mA_0.22_dm_model_core')
#df_decays = kinematic_diagnostic(masses=[2000], d=-7.5, r=458, tag='mDM_2000_mA_0.22_dm_model_floating_HIT_DETECTOR_TRACKER_VOL_COMBINED')
df_decays = kinematic_diagnostic(masses=[2000], d=-7.5, r=20, tag='mDM_2000_mA_0.22_dm_model_floating_HIT_DETECTOR_TRACKER_VOL_COMBINED')

In [ ]:
df_decays.columns

In [ ]:
df_decays['costh1'].hist(bins=100)

In [ ]:
df_decays['z0'].hist(bins=100)